In [ ]:
# this notebook conduct the following task:
# 1 testing protein complex information are from excel file, and DOWNLOAD tha pdb file with specific name format
# https://github.com/3BioCompBio/BCellEpitope/blob/main/List_nAbs_SARS-CoV-2.csv

# 2 Extract interaction info (tsv files) according to minumn atom-atom distance between sidechains of complexes
#   These tsv files have no SI information
#   structures_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/structures/"
#   output_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_info/"

# 3 Extract antigen's structure info and generate the mesh objs files using pymol
#   antigen_pdb_dir = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_pdbs'
#   antigen_obj_dir = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_objs'

# 4 add surface index to the antigen pdb file
#   SI_output_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_pdb_with_SI"

In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
## for the first-time execution of this code
## install the packages in google Drive locally (/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages/)
import os
import sys

nb_path = '/content/notebooks'

if not os.path.exists(nb_path):
    os.symlink('/content/drive/MyDrive/@antibody/antigen_epitope/abdb_v2/packages', nb_path)

sys.path.insert(0, nb_path)

In [ ]:
# first-time use execution
!pip install biopython --target=$nb_path

In [ ]:
!pip install scipy --target=$nb_path

In [ ]:
!pip install matplotlib ipywidgets --target=$nb_path

In [ ]:
!apt-get install -y python3-pymol

In [ ]:
!pip install ProDy

In [ ]:
!pip install freesasa

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/@antibody/antigen_epitope/abdb_v2/packages')

In [ ]:
import os
import numpy as np
import pandas as pd
from Bio import PDB

In [ ]:
from Bio.PDB import PDBParser, Select, PDBIO

class CleanPDB(Select):
    def accept_residue(self, residue):

        unwanted = ["HOH", "NA", "CA", "MG", "CL", "K", "ZN"]
        if residue.get_resname() in unwanted:
            return 0
        return 1

def clean_pdb(input_path, output_path):
    parser = PDBParser()
    structure = parser.get_structure("PDB_structure", input_path)

    io = PDBIO()
    io.set_structure(structure)
    io.save(output_path, CleanPDB())

input_pdb_path = '/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test_Cov/6vyb.pdb'
output_pdb_path = '/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test_Cov/6vyb_clean.pdb'

clean_pdb(input_pdb_path, output_pdb_path)
print(f"Processed and saved {output_pdb_path}")

In [ ]:
input_folder_path = "/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test/antigen_pdbs"

"/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test_Cov/SARS-CoV-2-nAbs/nAbs"

single_pdb_path = "/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test_Cov/6vyb_clean.pdb"

output_folder_path = "/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test_Cov/structures"

In [ ]:
import os
from Bio import PDB
from Bio import pairwise2
from Bio.PDB import PDBIO
import freesasa

# 这里添加刚才创建的pdb_chain_map
pdb_chain_map = {
 '7bz5': 'A',
 '7kmg': 'C',
 '7chb': 'R',
 '7chc_1': 'R',
 '7chc_2': 'R',
 '7ch4': 'R',
 '7k8m': 'E',
 '7kfw': 'A',
 '7kfx': 'A',
 '7c01': 'A',
 '6xc2': 'A',
 '6xc4': 'A',
 '7jmo': 'A',
 '7jmp': 'A',
 '6w41': 'C',
 '6xkq': 'A',
 '6xkp': 'A',
 '6xe1': 'E',
 '7k9z_1': 'E',
 '7k9z_2': 'E',
 '7l5b': 'A',
 '7klg': 'A',
 '7kmi': 'C',
 '7kmh': 'C',
 '7bwj': 'E',
 '7cdi': 'E',
 '7cjf': 'C',
 '7cm4': 'A',
 '7b3o': 'E',
 '7lm8_1': 'A',
 '7lm8_2': 'A',
 '7bel_1': 'R',
 '7bel_2': 'R',
 '7or9_1': 'E',
 '7eam': 'A',
 '7r6x_1': 'R',
 '7r6x_3': 'R',
 '7e88': 'C',
 '7e7y': 'R',
 '7m3i': 'C',
 '7mmo': 'C',
 '7cho': 'E',
 '7bep_1': 'E',
 '7bep_2': 'E',
 '7lop_1': 'A',
 '7beo_1': 'R',
 '7beo_2': 'R',
 '7r6w_1': 'R',
 '7m6d_1': 'C',
 '7kzb_1': 'C',
 '7m7w_1': 'R',
 '7m7w_2': 'R',
 '7jx3_3': 'R',
 '7jmw': 'A',
 '7neh': 'E',
 '7bei': 'E',
 '7chp': 'E',
 '7ean': 'A',
 '7djz': 'C',
 '7e86': 'C',
 '7chs': 'E',
 '7beh': 'E',
 '7dk0': 'C',
 '7bej': 'E',
 '7mf1': 'A',
 '7n3i': 'C',
 '7r8l_1': 'E',
 '7e7x': 'A',
 '7ly3': 'A',
 '7lxw': 'A',
 '7lxx': 'A',
 '7lab': 'C',
 '7k43': 'A',
 '7m71': 'A',
 '7czx': 'A',
 '7m7b': 'A',
 '7l2e': 'A',
 '7jv6': 'A',
 '7c2l': 'A',
 '7mkm': 'A',
 '7e8c_3': 'A',
 '7cwn_1': 'B',
 '7cwn_2': 'B'
}

def calculate_sasa(structure_file):
    structure = freesasa.Structure(structure_file)
    result = freesasa.calc(structure)
    sasa = { (residue.chain_id, residue.residue_number): residue.total_area
             for residue in result }
    return sasa

def is_epitope_residue(res_complex, res_spike, complex_asa, spike_asa, cutoff=5):
    if complex_asa is None or spike_asa is None:
        return False
    return (spike_asa - complex_asa) >= cutoff

def map_epitopes_to_6vyb(complex_dir, vyb_file, output_dir, pdb_chain_map):
    parser = PDB.PDBParser(QUIET=True)
    vyb_structure = parser.get_structure("6VYB", vyb_file)
    vyb_chain = vyb_structure[0]["A"]
    vyb_seq = "".join([res.get_resname() for res in vyb_chain if res.id[0] == ' '])

    vyb_sasa = calculate_sasa(vyb_file)

    vyb_epitopes = [False] * len(vyb_seq)

    for pdb_file in os.listdir(complex_dir):
        if pdb_file.endswith(".pdb"):
            pdb_code = pdb_file[:-4]
            complex_structure = parser.get_structure(pdb_code, os.path.join(complex_dir, pdb_file))
            complex_sasa = calculate_sasa(os.path.join(complex_dir, pdb_file))
            chain_id = pdb_chain_map.get(pdb_code, 'A')

            antigen_chain = complex_structure[0][chain_id]
            antigen_seq = "".join([res.get_resname() for res in antigen_chain if res.id[0] == ' '])

            for res_complex in antigen_chain:
                if res_complex.id[0] != ' ':
                    continue
                res_spike_index = vyb_chain.index(res_complex)
                res_spike = vyb_chain[res_spike_index]

                complex_res_asa = complex_sasa.get((chain_id, res_complex.id[1]), None)
                spike_res_asa = vyb_sasa.get(('A', res_spike.id[1]), None)

                if is_epitope_residue(res_complex, res_spike, complex_res_asa, spike_res_asa):
                    vyb_epitopes[res_spike_index] = True

    pdbio = PDB.PDBIO()
    pdbio.set_structure(vyb_structure)
    output_file = os.path.join(output_dir, "6VYB_epitopes.pdb")
    pdbio.save(output_file, select=lambda atom: vyb_epitopes[atom.get_parent().id[1] - 1])

    return vyb_epitopes



complex_dir = "/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test_Cov/SARS-CoV-2-nAbs/nAbs"
vyb_file = "/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test_Cov/6vyb.pdb"
output_dir = "/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test_Cov/structures"
map_epitopes_to_6vyb(complex_dir, vyb_file, output_dir, pdb_chain_map)

In [ ]:
import os

def delete_files(directory, prefix):
    for filename in os.listdir(directory):
        if filename.startswith(prefix):
            file_path = os.path.join(directory, filename)
            os.remove(file_path)
            print(f"Deleted: {file_path}")

delete_files("/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test_Cov/structures", 'aligned')

In [ ]:
from Bio import PDB

parser = PDB.PDBParser()
structure = parser.get_structure('6VYB', "/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test_Cov/6vyb.pdb")
for model in structure:
    for chain in model:
        print(f"Chain ID: {chain.id}")

In [ ]:
#@title read the excel file List_nAbs_SARS-CoV-2
import pandas as pd

# Update the path to reflect the exact location of your Excel file
file_path = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data_test/List_nAbs_SARS-CoV-2.csv'
df = pd.read_csv(file_path)

# Set display options
pd.set_option('display.max_columns', None)  # No limit on the number of columns
pd.set_option('display.max_colwidth', None)  # No limit on the column width
pd.set_option('display.width', 1000)  # Adjust the total width of the display

# Display the DataFrame
print(df)

In [ ]:
#@title modify the PDBCode

# Function to update the 'PDBCode' column
def update_pdb_code(code):
    # Check if code already ends with '_1', '_2', or '_3'
    if code.endswith(('_1', '_2', '_3')):
        return code
    else:
        # Otherwise, append '_1'
        return code[:4] + '_1'

# Apply the function to the 'PDBCode' column
df['PDBCode'] = df['PDBCode'].apply(update_pdb_code)

# Display the modified DataFrame to verify changes
print(df)

In [ ]:
# Save the modified DataFrame back to the same directory with a new name to avoid overwriting the original
new_file_path = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data_test/modified_csv.csv'
df.to_csv(new_file_path, index=False)

# Confirmation message
print('DataFrame saved successfully at:', new_file_path)

In [ ]:
#@title DOWNLOAD the antibody-antigen complex from data bank, and saved as specific name format, e.g. 6w41_1_H_L_C.pdb
import pandas as pd
from Bio.PDB import PDBList, PDBParser, Select, PDBIO

# Define a class to select only the specific chains
class ChainSelect(Select):
    def __init__(self, chains):
        self.chains = chains

    def accept_chain(self, chain):
        if chain.get_id() in self.chains:
            return 1
        else:
            return 0

    def accept_residue(self, residue):
        # Accept only amino acids, no water, ions, or other heteroatoms
        return 1 if residue.id[0] == ' ' else 0

# Initialize PDB downloader and parser
pdbl = PDBList()
parser = PDBParser()
io = PDBIO()


# Directory to save the modified PDB files
output_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v2/data_test_Cov/'

for index, row in df.iterrows():
    pdb_id = row['PDBCode'][:4]  # Extract the PDB ID from the first four characters
    heavy_chain_id = row['HL'][0]  # Extract heavy chain ID (first character of 'HL')
    light_chain_id = row['HL'][1]  # Extract light chain ID (second character of 'HL')
    antigen_id = row['Chain Spike']  # Antigen chain ID
    chains = [heavy_chain_id, light_chain_id, antigen_id]  # List of chains: heavy, light, and antigen

    # File paths for downloading and saving
    file_path = pdbl.retrieve_pdb_file(pdb_id, pdir=output_dir, file_format='pdb')
    structure = parser.get_structure(pdb_id, file_path)

    # Set up to save the selected chains only
    chain_selector = ChainSelect(chains)
    io.set_structure(structure)
    output_filename = f"{row['PDBCode']}_{heavy_chain_id}_{light_chain_id}_{antigen_id}.pdb"
    io.save(f"{output_dir}{output_filename}", select=chain_selector)

    print(f"Processed and saved {output_filename}")

In [ ]:
import os

# Directory to check
output_dir = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data_test/structures/'

# Check if the directory exists
if os.path.exists(output_dir):
    # List all files in the directory
    files = os.listdir(output_dir)
    print(f"There are {len(files)} files in the directory:")
    for file_name in files:
        print(file_name)
else:
    print("Directory does not exist. Please check the path.")

In [ ]:
#@title Extract antigen's structure info and interaction info ==> minumn atom-atom distance between sidechains

from Bio import PDB
import pandas as pd
from pymol import cmd, stored
from scipy.spatial import distance
import os

# Function to calculate center of mass of sidechain
# Glycine (GLY) does not have a side chain beyond its alpha carbon, so its side chain center of mass would be undefined in your code
def calculate_center_of_mass(residue):
    if residue.resname == "GLY":
        return residue['CA'].coord  # Return alpha carbon coordinates for Glycine

    mass = 0
    x, y, z = 0, 0, 0
    backbone_atoms = ['N', 'CA', 'C', 'O']
    for atom in residue.get_unpacked_list():
        if atom.name not in backbone_atoms:
            mass += atom.mass
            x += atom.coord[0] * atom.mass
            y += atom.coord[1] * atom.mass
            z += atom.coord[2] * atom.mass
    if mass == 0:
        return 0, 0, 0
    return x / mass, y / mass, z / mass

# Function to check for heavy atom-atom interactions
def check_atom_atom_interaction(residue1, residue2, cutoff_distance=4.5, ca_cutoff_distance=15):
    # To save computational time, Check the CA-CA distance first
    if "CA" in residue1 and "CA" in residue2:
        ca_distance = distance.euclidean(residue1["CA"].coord, residue2["CA"].coord)
        if ca_distance > ca_cutoff_distance:
            return False  # If the CA-CA distance is greater than the threshold, no need to check atom-atom distances

    # Proceed with atom-atom distance calculations
    for atom1 in residue1.get_unpacked_list():
        if atom1.element == "H":  # skip hydrogen atoms
            continue
        for atom2 in residue2.get_unpacked_list():
            if atom2.element == "H":  # skip hydrogen atoms
                continue
            if distance.euclidean(atom1.coord, atom2.coord) <= cutoff_distance:
                return True  # found an interaction
    return False

# Process each PDB file in the directory
structures_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data_test/structures/"
output_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data_test/antigen_info/"
os.makedirs(output_dir, exist_ok=True)

for pdb_file in os.listdir(structures_dir):
    if pdb_file.endswith('.pdb'):
        pdb_file_path = os.path.join(structures_dir, pdb_file)
        file_identifier = pdb_file[:-4]  # Remove the .pdb extension
        #print(file_identifier)  # For debugging

        # Extract chain information
        parts = file_identifier.split('_')
        pdb_id = parts[0]
        chains = parts[2:-1]  # Antibody chains are typically L and H
        antigen_chain = parts[-1]  # Antigen chains are the remaining parts

        # Load the PDB file into PyMOL
        cmd.reinitialize()  # Clear previous data
        cmd.load(pdb_file_path, "protein_complex")

        # Calculate relative SASA in the presence of antibody
        # SaSaRate = cmd.get_sasa_relative("protein_complex and chain " + antigen_chain)
        # sasa_values = [value for key, value in SaSaRate.items()]

        # Initialize an empty list for secondary structure
        stored.ss_values = []
        cmd.iterate("protein_complex and chain " + antigen_chain + " and name CA", "stored.ss_values.append(ss)")

         # Hide or remove the antibody chains
        cmd.remove("chain " + "+".join(chains))  # Using the `remove` command to delete the antibody chains
        # Calculate relative SASA for the antigen chain in the absence of the antibody
        SaSaRate = cmd.get_sasa_relative("protein and chain " + antigen_chain)
        sasa_values = [value for key, value in SaSaRate.items()]

        # Initialize PDB Parser
        parser = PDB.PDBParser(QUIET=True)

        # Load structure
        structure = parser.get_structure(pdb_file.split('.')[0], pdb_file_path)

        # Initialize DataFrame
        columns = ['name', 'ResID', 'AA', 'SS', 'Cax', 'Cay', 'Caz', 'SCx', 'SCy', 'SCz', 'SaSaRate',
                   'G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']
        df = pd.DataFrame(columns=columns)

        # Prepare a list for antibody residues
        antibody_residues = []
        for model in structure:
            for chain in model:
                if chain.id in chains:
                    for residue in chain:
                        # Ensure that the residue is an amino acid
                        if residue.resname in PDB.Polypeptide.protein_letters_3to1:
                            antibody_residues.append(residue)

        # Iterate through structure to get antigen residues
        for model in structure:
            for chain in model:
                if chain.id == antigen_chain:
                    for i, residue in enumerate(chain):
                        if "CA" in residue:
                            c_alpha = residue['CA'].coord
                            # Safeguard against IndexError
                            ss = stored.ss_values[i] if i < len(stored.ss_values) else 'Unknown'
                            sc_x, sc_y, sc_z = calculate_center_of_mass(residue)

                            interaction_data = {aa: 0 for aa in ['G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']}
                            for ab_residue in antibody_residues:
                                if check_atom_atom_interaction(residue, ab_residue):
                                    one_letter_code = PDB.Polypeptide.protein_letters_3to1[ab_residue.resname]
                                    interaction_data[one_letter_code] = 1

                            data = {'name': pdb_file.split('.')[0],
                                    'ResID': residue.id[1],  # Residue number
                                    'AA': residue.resname,
                                    'SS': ss,
                                    'Cax': c_alpha[0],
                                    'Cay': c_alpha[1],
                                    'Caz': c_alpha[2],
                                    'SCx': sc_x,
                                    'SCy': sc_y,
                                    'SCz': sc_z,
                                    'SaSaRate': sasa_values[i]}
                            data.update(interaction_data)
                            df = pd.concat([df, pd.DataFrame([data])], ignore_index=True)

        # Save DataFrame to TSV
        output_path = os.path.join(output_dir, f"{pdb_file.split('.')[0]}_info.tsv")
        df.to_csv(output_path, sep='\t', index=False)

        print(f"Processed {pdb_file} and saved to {output_path}")

In [ ]:
#@title extract antigen chains and save

import os
import shutil
from Bio import PDB

# Define the source and target directories
source_directory = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data_test/structures'
target_directory = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data_test/antigen_pdbs'

# Create the target directory if it doesn't exist
os.makedirs(target_directory, exist_ok=True)

# Initialize the PDB parser
parser = PDB.PDBParser()
io = PDB.PDBIO()

# Function to extract and save antigen chains
def save_antigen_chains(structure, source_filename, target_directory):
    for model in structure:
        # Create a new structure for antigens
        antigen_structure = PDB.Structure.Structure('antigen')

        # Copy model to antigen structure
        antigen_model = model.copy()
        antigen_structure.add(antigen_model)
        antigen_model.detach_parent()

        # Remove antibody chains (L and H) from antigen model
        for chain in list(antigen_model.get_chains()):
            if chain.id in ['L', 'H']:
                antigen_model.detach_child(chain.id)

        # If there are chains left (antigen), save the structure
        if len(list(antigen_model.get_chains())) > 0:
            new_filename = f"{source_filename[:-4]}_antigen.pdb"
            target_filepath = os.path.join(target_directory, new_filename)
            io.set_structure(antigen_structure)
            io.save(target_filepath)
            print(f"Saved antigen chains to {new_filename}")

# Iterate over PDB files and process them
for filename in os.listdir(source_directory):
    if filename.endswith(".pdb"):
        source_filepath = os.path.join(source_directory, filename)
        structure = parser.get_structure('PDB_structure', source_filepath)
        save_antigen_chains(structure, filename, target_directory)


In [ ]:
#@title Generate antigen mesh obj files
# Create .obj file for each antigen pdb (run this code if antigen objs are not created)

from pymol import cmd

antigen_pdb_dir = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data_test/antigen_pdbs'
antigen_obj_dir = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data_test/antigen_objs'

if not os.path.exists(antigen_obj_dir):
  os.makedirs(antigen_obj_dir)
  print(f"Folder '{antigen_obj_dir}' was created.")
else:
  print(f"Folder '{antigen_obj_dir}' already exists.")

for pdb_filename in os.listdir(antigen_pdb_dir):
  if pdb_filename.endswith('.pdb'):
    # reset pymol
    cmd.reinitialize()

    pdb_file_path = os.path.join(antigen_pdb_dir, pdb_filename)

    # load pdb
    cmd.load(pdb_file_path, 'pdb_filename')
    cmd.hide('everything', 'pdb_filename')
    cmd.show('surface', 'pdb_filename')

    # set camera view
    cmd.set_view((
        1,    0,    0,
        0,    1,    0,
        0,    0,    1,
        0,    0,    0,
        0,    0,    0,
        0,    300,  1
    ))

    # save obj file
    obj_filename = pdb_filename.replace('.pdb', '.obj')
    cmd.save(os.path.join(antigen_obj_dir, obj_filename), 'pdb_filename', format='obj')


In [ ]:
#@title Generate antigen pdb with SI

In [ ]:
!pip install pywavefront
!pip install ipyvolume
!pip install pyvista
!pip install matplotlib ipywidgets

In [ ]:
!pip install trimesh
!pip install rtree

In [ ]:
import ipyvolume as ipv
from matplotlib import cm
import pyvista as pv
import pywavefront
import os

In [ ]:
# dir to save info with SI
SI_output_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data_test/antigen_pdb_with_SI"
antigen_pdb_dir = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data_test/antigen_pdbs'
antigen_obj_dir = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data_test/antigen_objs'

In [ ]:
# Function to count files
def count_files(directory):
    return len([name for name in os.listdir(directory) if os.path.isfile(os.path.join(directory, name))])

# Count files in each directory
SI_output_count = count_files(SI_output_dir)
antigen_obj_count = count_files(antigen_obj_dir)
antigen_pdb_count = count_files(antigen_pdb_dir)

print(f"Files in SI output directory: {SI_output_count}")
print(f"Files in antigen objects directory: {antigen_obj_count}")
print(f"Files in antigen PDB directory: {antigen_pdb_count}")

In [ ]:
#@title Add Shape Index (yuhao method 4)
import numpy as np
import trimesh
import os
import pandas as pd
from Bio import PDB
from scipy.spatial import KDTree

for obj_filename in os.listdir(antigen_obj_dir):
    if obj_filename.endswith('.obj'):
        mesh = trimesh.load_mesh(os.path.join(antigen_obj_dir, obj_filename))
        vertices = mesh.vertices
        faces = mesh.faces

        K = trimesh.curvature.discrete_gaussian_curvature_measure(mesh, mesh.vertices, 2)
        H = trimesh.curvature.discrete_mean_curvature_measure(mesh, mesh.vertices, 2)
        elem = np.square(H) - K
        elem[elem <= 0] = 1e-8
        k1 = H + np.sqrt(elem)
        k2 = H - np.sqrt(elem)
        shape_index = (2.0 / np.pi) * np.arctan((k1 + k2) / (k1 - k2))

        obj_coords = mesh.vertices
        obj_curvature = shape_index

        pdb_filename = os.path.splitext(obj_filename)[0] + '.pdb'
        pdb_filepath = os.path.join(antigen_pdb_dir, pdb_filename)

        parser = PDB.PDBParser()
        structure = parser.get_structure("protein", pdb_filepath)

        pdb_coords = []
        for atom in structure.get_atoms():
            pdb_coords.append(atom.get_coord())
        pdb_coords = np.array(pdb_coords)

        kd_tree = KDTree(obj_coords)
        distances, indices = kd_tree.query(pdb_coords)

        atom_index = 0
        for atom in structure.get_atoms():
            atom.set_bfactor(shape_index[indices[atom_index]])
            atom_index += 1

        output_pdb_filename = os.path.splitext(pdb_filename)[0] + '_updated.pdb'
        output_pdb_filepath = os.path.join(SI_output_dir, output_pdb_filename)

        io = PDB.PDBIO()
        io.set_structure(structure)
        io.save(output_pdb_filepath)

In [ ]:
# open the file locally with pymol
# check the SI using this code
# as surface spectrum b, blue_white_red, minimum=-1, maximum=1